In [3]:
pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 15.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
"""
Steam 개인 맞춤 게임 추천 - LightGBM

사용 데이터
1. steam_user_games_merged.csv
   - steamid
   - game_name
   - playtime_hours
   - genre
   - tags

2. steam_top500_games.csv
   - appid
   - game_name
   - rating
   - genre
   - tags

추가
- Steam Web API GetOwnedGames
  → 사용자의 현재 전체 게임 라이브러리 확인
  → 이미 보유한 게임 추천에서 제외

처리 과정

사용자 게임
    ↓
플레이타임 기반 가중치
    ↓
사용자 장르 선호도
    ↓
게임 Feature Vector
    ↓
학습 데이터 생성
    ↓
LightGBM
    ↓
좋아할 확률 예측
    ↓
Steam 전체 라이브러리와 비교
    ↓
이미 보유한 게임 제거
    ↓
추천 TOP N
"""


import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
import re
import warnings
import requests
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

from lightgbm import LGBMClassifier


warnings.filterwarnings("ignore")


# ============================================================
# 0. 파일 설정
# ============================================================

USER_CSV = str(Path(__file__).resolve().parents[2] / "data" / "processed" / "steam_user_games_classified.csv") if "__file__" in dir() else "../../data/processed/steam_user_games_classified.csv"

TOP500_CSV = "../../data/processed/steam_top500_games_classified.csv"

OUTPUT_CSV = "../../reports/steam_lightgbm_recommendations.csv"


# ============================================================
# Steam API
# ============================================================

STEAM_API_KEY = os.getenv("STEAM_API_KEY", "")


# ============================================================
# 1. 컬럼 설정
# ============================================================

COL_STEAM_ID = "steamid"
COL_GAME = "game_name"
COL_PLAYTIME = "playtime_hours"
COL_GENRE = "genre"
COL_TAGS = "tags"


# ============================================================
# 2. 장르 분류 기준
# ============================================================

GENRE_KEYWORDS = {

    "Action": [
        "action",
        "hack and slash",
        "beat 'em up",
        "beat em up",
        "fighting",
        "martial arts",
        "character action"
    ],

    "Adventure": [
        "adventure",
        "point & click",
        "point and click",
        "exploration",
        "walking simulator",
        "story rich",
        "interactive fiction"
    ],

    "RPG": [
        "rpg",
        "action rpg",
        "action-rpg",
        "jrpg",
        "crpg",
        "souls-like",
        "soulslike",
        "party-based rpg",
        "turn-based rpg",
        "turn based rpg",
        "old school rpg",
        "roguelike rpg",
        "role-playing"
    ],

    "Strategy": [
        "strategy",
        "real-time strategy",
        "real time strategy",
        "rts",
        "turn-based strategy",
        "turn based strategy",
        "4x",
        "grand strategy",
        "tactics",
        "tactical",
        "tower defense",
        "tower defence"
    ],

    "Simulation": [
        "simulation",
        "sim",
        "life sim",
        "farming sim",
        "city builder",
        "management",
        "building",
        "automation",
        "sandbox"
    ],

    "Shooter": [
        "shooter",
        "fps",
        "first-person shooter",
        "first person shooter",
        "third-person shooter",
        "third person shooter",
        "tps",
        "tactical shooter",
        "hero shooter",
        "looter shooter",
        "bullet hell"
    ],

    "Sports": [
        "sports",
        "football",
        "soccer",
        "basketball",
        "baseball",
        "tennis",
        "golf",
        "hockey",
        "volleyball",
        "wrestling"
    ],

    "Racing": [
        "racing",
        "racer",
        "driving",
        "automobile sim",
        "car"
    ],

    "Horror": [
        "horror",
        "survival horror",
        "psychological horror"
    ],

    "Survival": [
        "survival",
        "survival crafting",
        "open world survival craft",
        "crafting",
        "base building"
    ],

    "Platformer": [
        "platformer",
        "2d platformer",
        "3d platformer",
        "metroidvania"
    ],

    "Puzzle": [
        "puzzle",
        "logic",
        "match 3",
        "match-3"
    ],

    "Casual": [
        "casual",
        "relaxing",
        "family friendly"
    ]
}


GENRES = list(
    GENRE_KEYWORDS.keys()
)


# ============================================================
# 3. 문자열 처리
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value


def split_values(value):

    if value is None:
        return []

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    if value.lower() == "unknown":
        return []

    result = []

    for item in value.split(","):

        item = item.strip()

        if item:
            result.append(item)

    return result


# ============================================================
# 4. 장르 탐지
# ============================================================

def detect_genres(values):

    detected = []

    for value in values:

        value = normalize_text(value)

        if not value:
            continue

        for genre, keywords in GENRE_KEYWORDS.items():

            for keyword in keywords:

                keyword = normalize_text(
                    keyword
                )

                if (
                    value == keyword
                    or keyword in value
                ):

                    if genre not in detected:

                        detected.append(
                            genre
                        )

                    break

    return detected


def determine_genres(
    genre,
    tags
):

    # --------------------------------------------------------
    # Genre 우선
    # --------------------------------------------------------

    genre_values = split_values(
        genre
    )

    detected = detect_genres(
        genre_values
    )

    if detected:

        return detected


    # --------------------------------------------------------
    # Genre가 없으면 Tags
    # --------------------------------------------------------

    tag_values = split_values(
        tags
    )

    detected = detect_genres(
        tag_values
    )

    if detected:

        return detected


    return ["Unknown"]


# ============================================================
# 5. 게임 Feature Vector
# ============================================================

def make_game_vector(
    genre,
    tags
):

    detected_genres = determine_genres(
        genre,
        tags
    )

    vector = {}

    for g in GENRES:

        if g in detected_genres:

            vector[g] = 1

        else:

            vector[g] = 0

    return vector


# ============================================================
# 6. 데이터 로드
# ============================================================

def load_data():

    print("=" * 70)

    print(
        "CSV 파일 로딩"
    )

    print("=" * 70)


    user_df = pd.read_csv(

        USER_CSV,

        dtype={
            COL_STEAM_ID: str
        }

    )


    top500_df = pd.read_csv(

        TOP500_CSV,

        dtype={
            "appid": str
        }

    )


    print(
        f"사용자 데이터: "
        f"{len(user_df):,}개"
    )

    print(
        f"TOP500 데이터: "
        f"{len(top500_df):,}개"
    )


    return (
        user_df,
        top500_df
    )


# ============================================================
# 7. 사용자 게임 추출
# ============================================================

def get_user_games(
    user_df,
    steam_id
):

    steam_id = str(
        steam_id
    ).strip()


    df = user_df[

        user_df[
            COL_STEAM_ID
        ]
        .astype(str)
        .str.strip()
        == steam_id

    ].copy()


    if df.empty:

        raise ValueError(

            f"Steam ID '{steam_id}'의 "
            "데이터를 찾을 수 없습니다."

        )


    # --------------------------------------------------------
    # 플레이타임 숫자 변환
    # --------------------------------------------------------

    df[
        COL_PLAYTIME
    ] = pd.to_numeric(

        df[
            COL_PLAYTIME
        ],

        errors="coerce"

    )


    df = df.dropna(

        subset=[
            COL_PLAYTIME
        ]

    )


    # --------------------------------------------------------
    # 플레이타임 > 0
    # --------------------------------------------------------

    df = df[
        df[
            COL_PLAYTIME
        ] > 0
    ].copy()


    # --------------------------------------------------------
    # 게임별 중복 제거
    # --------------------------------------------------------

    df = (

        df

        .groupby(

            [
                COL_GAME,
                COL_GENRE,
                COL_TAGS
            ],

            dropna=False,

            as_index=False

        )[COL_PLAYTIME]

        .sum()

    )


    return df


# ============================================================
# 8. Steam 전체 라이브러리 가져오기
# ============================================================

def get_steam_library(
    steam_id
):

    print()
    print("=" * 70)

    print(
        "Steam 전체 게임 라이브러리 확인"
    )

    print("=" * 70)


    url = (
        "https://api.steampowered.com/"
        "IPlayerService/GetOwnedGames/v1/"
    )


    params = {

        "key":
            STEAM_API_KEY,

        "steamid":
            steam_id,

        "include_appinfo":
            1,

        "include_played_free_games":
            1

    }


    try:

        response = requests.get(

            url,

            params=params,

            timeout=20

        )


        response.raise_for_status()


        data = response.json()


    except Exception as e:

        print()
        print(
            "Steam API 요청 실패:"
        )

        print(e)

        return None


    response_data = data.get(
        "response",
        {}
    )


    games = response_data.get(
        "games",
        []
    )


    # --------------------------------------------------------
    # Steam API 응답에 games가 없는 경우
    # --------------------------------------------------------

    if "games" not in response_data:

        print()
        print(
            "Steam 전체 라이브러리를 "
            "가져오지 못했습니다."
        )

        print(
            "프로필 공개 여부와 "
            "Steam API Key를 확인해주세요."
        )

        return None


    owned_games = set()


    for game in games:

        name = game.get(
            "name",
            ""
        )


        if name:

            owned_games.add(

                normalize_text(
                    name
                )

            )


    print(
        f"Steam 전체 라이브러리: "
        f"{len(owned_games):,}개"
    )


    return owned_games


# ============================================================
# 9. 플레이타임 가중치
# ============================================================

def calculate_weights(
    user_games
):

    df = user_games.copy()


    playtime = (

        df[
            COL_PLAYTIME
        ]
        .astype(float)
        .to_numpy()

    )


    log_playtime = np.log1p(
        playtime
    )


    total = log_playtime.sum()


    if total == 0:

        df[
            "weight"
        ] = 1 / len(df)

    else:

        df[
            "weight"
        ] = (

            log_playtime
            / total

        )


    return df


# ============================================================
# 10. 사용자 선호도
# ============================================================

def build_user_preference(
    weighted_games
):

    preference = {

        genre: 0.0

        for genre in GENRES

    }


    for _, row in weighted_games.iterrows():

        genres = determine_genres(

            row[
                COL_GENRE
            ],

            row[
                COL_TAGS
            ]

        )


        weight = float(
            row["weight"]
        )


        for genre in genres:

            if genre in preference:

                preference[
                    genre
                ] += weight


    total = sum(
        preference.values()
    )


    if total > 0:

        for genre in preference:

            preference[
                genre
            ] /= total


    return preference


# ============================================================
# 11. TOP500 Feature 생성
# ============================================================

def create_game_features(
    top500_df
):

    rows = []


    for _, row in top500_df.iterrows():

        vector = make_game_vector(

            row.get(
                COL_GENRE,
                ""
            ),

            row.get(
                COL_TAGS,
                ""
            )

        )


        result = {

            "appid":
                str(row["appid"]),

            "game_name":
                row[COL_GAME]

        }


        for genre in GENRES:

            result[
                f"game_{genre}"
            ] = vector[
                genre
            ]


        rating = pd.to_numeric(

            row.get(
                "rating",
                np.nan
            ),

            errors="coerce"

        )


        if pd.isna(rating):

            rating = 0


        result[
            "rating"
        ] = float(rating)


        rows.append(
            result
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 12. 학습 데이터 생성
# ============================================================

def create_training_data(
    user_games,
    game_features
):

    played_games = set(

        user_games[
            COL_GAME
        ]
        .astype(str)
        .str.lower()
        .str.strip()

    )


    X = []

    y = []


    # --------------------------------------------------------
    # Positive
    # --------------------------------------------------------

    for _, game in game_features.iterrows():

        game_name = normalize_text(

            game[
                "game_name"
            ]

        )


        if game_name not in played_games:

            continue


        feature = []


        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        feature.append(
            game["rating"]
        )


        X.append(
            feature
        )

        y.append(1)


    # --------------------------------------------------------
    # Negative
    # --------------------------------------------------------

    for _, game in game_features.iterrows():

        game_name = normalize_text(

            game[
                "game_name"
            ]

        )


        if game_name in played_games:

            continue


        feature = []


        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        feature.append(
            game["rating"]
        )


        X.append(
            feature
        )

        y.append(0)


    return (

        np.array(
            X,
            dtype=float
        ),

        np.array(
            y,
            dtype=int
        )

    )


# ============================================================
# 13. LightGBM 학습
# ============================================================

def train_lightgbm(
    X,
    y
):

    print()
    print("=" * 70)

    print(
        "LightGBM 학습"
    )

    print("=" * 70)


    print(
        f"학습 데이터: "
        f"{len(X):,}개"
    )

    print(
        f"Positive: "
        f"{sum(y == 1):,}개"
    )

    print(
        f"Negative: "
        f"{sum(y == 0):,}개"
    )


    # --------------------------------------------------------
    # Train / Test
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = (

        train_test_split(

            X,

            y,

            test_size=0.2,

            random_state=42,

            stratify=y

        )

    )


    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    model = LGBMClassifier(

        n_estimators=300,

        learning_rate=0.05,

        max_depth=10,

        num_leaves=31,

        min_child_samples=10,

        subsample=0.8,

        colsample_bytree=0.8,

        class_weight="balanced",

        objective="binary",

        random_state=42,

        n_jobs=-1,

        verbosity=-1

    )


    model.fit(

        X_train,

        y_train

    )


    # --------------------------------------------------------
    # 예측
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test
    )


    y_prob = model.predict_proba(

        X_test

    )[:, 1]


    # ========================================================
    # 평가
    # ========================================================

    accuracy = accuracy_score(

        y_test,
        y_pred

    )


    precision = precision_score(

        y_test,
        y_pred,

        zero_division=0

    )


    recall = recall_score(

        y_test,
        y_pred,

        zero_division=0

    )


    f1 = f1_score(

        y_test,
        y_pred,

        zero_division=0

    )


    try:

        roc_auc = roc_auc_score(

            y_test,
            y_prob

        )

    except:

        roc_auc = 0


    try:

        pr_auc = average_precision_score(

            y_test,
            y_prob

        )

    except:

        pr_auc = 0


    print()
    print(
        "=== LightGBM 평가 ==="
    )


    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1-score : {f1:.4f}"
    )

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC   : {pr_auc:.4f}"
    )


    print()
    print(
        "=== Classification Report ==="
    )


    print(

        classification_report(

            y_test,

            y_pred,

            zero_division=0

        )

    )


    # ========================================================
    # Feature Importance
    # ========================================================

    print()
    print(
        "=== Feature Importance ==="
    )


    feature_names = [

        f"game_{genre}"

        for genre in GENRES

    ] + ["rating"]


    importance_df = pd.DataFrame({

        "feature":
            feature_names,

        "importance":
            model.feature_importances_

    })


    importance_df = (

        importance_df

        .sort_values(

            "importance",

            ascending=False

        )

    )


    print(

        importance_df.to_string(
            index=False
        )

    )


    return model


# ============================================================
# 14. 추천 게임 계산
# ============================================================

def recommend_games(

    model,

    game_features,

    user_preference,

    steam_library,

    top_n=20

):

    df = game_features.copy()


    # --------------------------------------------------------
    # Feature 생성
    # --------------------------------------------------------

    X_recommend = []


    for _, game in df.iterrows():

        feature = []


        for genre in GENRES:

            feature.append(

                game[
                    f"game_{genre}"
                ]

            )


        feature.append(
            game["rating"]
        )


        X_recommend.append(
            feature
        )


    X_recommend = np.array(

        X_recommend,

        dtype=float

    )


    # --------------------------------------------------------
    # LightGBM 예측
    # --------------------------------------------------------

    probabilities = (

        model

        .predict_proba(

            X_recommend

        )[:, 1]

    )


    df[
        "lightgbm_probability"
    ] = probabilities


    # --------------------------------------------------------
    # 사용자 장르 선호도
    # --------------------------------------------------------

    preference_scores = []


    for _, game in df.iterrows():

        score = 0.0


        for genre in GENRES:

            game_value = game[

                f"game_{genre}"

            ]


            user_value = (

                user_preference[
                    genre
                ]

            )


            score += (

                game_value
                * user_value

            )


        preference_scores.append(
            score
        )


    df[
        "genre_preference_score"
    ] = preference_scores


    # --------------------------------------------------------
    # 최종 점수
    # --------------------------------------------------------

    df[
        "final_score"
    ] = df[
        "lightgbm_probability"
    ]


    # --------------------------------------------------------
    # 게임 이름 정규화
    # --------------------------------------------------------

    df[
        "game_name_lower"
    ] = (

        df[
            "game_name"
        ]

        .astype(str)

        .str.lower()

        .str.strip()

    )


    # ========================================================
    # 이미 보유한 게임 제외
    # ========================================================

    before_count = len(df)


    owned_mask = df[
        "game_name_lower"
    ].isin(
        steam_library
    )


    excluded_count = int(
        owned_mask.sum()
    )


    df = df[
        ~owned_mask
    ].copy()


    after_count = len(df)


    print()
    print("=" * 70)
    print("이미 보유한 게임 제외")
    print("=" * 70)

    print(
        f"추천 후보 게임: "
        f"{before_count}개"
    )

    print(
        f"보유 게임으로 제외: "
        f"{excluded_count}개"
    )

    print(
        f"최종 추천 후보: "
        f"{after_count}개"
    )


    # --------------------------------------------------------
    # 추천 순위
    # --------------------------------------------------------

    df = (

        df

        .sort_values(

            "final_score",

            ascending=False

        )

        .head(top_n)

        .reset_index(
            drop=True
        )

    )


    df[
        "rank"
    ] = np.arange(

        1,

        len(df) + 1

    )


    return df


# ============================================================
# 15. 메인
# ============================================================

def main():

    print("=" * 70)

    print(
        "Steam LightGBM 개인 맞춤 추천 시스템"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Steam ID
    # --------------------------------------------------------

    steam_id = input(

        "Steam ID를 입력하세요: "

    ).strip()


    # --------------------------------------------------------
    # 데이터 로드
    # --------------------------------------------------------

    user_df, top500_df = load_data()


    # --------------------------------------------------------
    # 사용자 플레이 데이터
    # --------------------------------------------------------

    user_games = get_user_games(

        user_df,

        steam_id

    )


    print()

    print(

        f"사용자 플레이 데이터: "
        f"{len(user_games):,}개"

    )


    # --------------------------------------------------------
    # Steam 전체 라이브러리
    # --------------------------------------------------------

    steam_library = get_steam_library(

        steam_id

    )


    # --------------------------------------------------------
    # Steam 라이브러리를 가져오지 못하면 중단
    # --------------------------------------------------------

    if steam_library is None:

        raise ValueError(

            "Steam 전체 라이브러리를 "
            "확인할 수 없어 추천을 중단합니다.\n"
            "Steam API Key와 프로필 공개 여부를 "
            "확인해주세요."

        )


    # --------------------------------------------------------
    # 플레이타임 가중치
    # --------------------------------------------------------

    weighted_games = (

        calculate_weights(

            user_games

        )

    )


    # --------------------------------------------------------
    # 사용자 취향
    # --------------------------------------------------------

    user_preference = (

        build_user_preference(

            weighted_games

        )

    )


    print()
    print(
        "=== 사용자 장르 선호도 ==="
    )


    for genre, value in sorted(

        user_preference.items(),

        key=lambda x: x[1],

        reverse=True

    ):

        print(

            f"{genre:15s}: "
            f"{value:.4f}"

        )


    # --------------------------------------------------------
    # TOP500 Feature
    # --------------------------------------------------------

    game_features = (

        create_game_features(

            top500_df

        )

    )


    # --------------------------------------------------------
    # 학습 데이터
    # --------------------------------------------------------

    X, y = (

        create_training_data(

            user_games,

            game_features

        )

    )


    if len(X) < 10:

        raise ValueError(

            "학습 데이터가 너무 적습니다."

        )


    if len(
        np.unique(y)
    ) < 2:

        raise ValueError(

            "Positive와 Negative "
            "데이터가 둘 다 필요합니다."

        )


    # --------------------------------------------------------
    # LightGBM 학습
    # --------------------------------------------------------

    model = train_lightgbm(

        X,

        y

    )


    # --------------------------------------------------------
    # 추천
    # --------------------------------------------------------

    recommendations = (

        recommend_games(

            model,

            game_features,

            user_preference,

            steam_library,

            top_n=20

        )

    )


    # --------------------------------------------------------
    # 결과
    # --------------------------------------------------------

    print()
    print()

    print("=" * 70)

    print(
        "=== 개인 맞춤 추천 TOP 20 ==="
    )

    print("=" * 70)


    for _, row in recommendations.iterrows():

        print()

        print(

            f"{int(row['rank']):2d}. "
            f"{row['game_name']}"

        )

        print(

            f"    AppID: "
            f"{row['appid']}"

        )

        print(

            f"    LightGBM "
            f"좋아할 확률: "
            f"{row['lightgbm_probability']:.2%}"

        )

        print(

            f"    장르 선호도: "
            f"{row['genre_preference_score']:.4f}"

        )

        print(

            f"    평점: "
            f"{row['rating']:.2f}"

        )


    # --------------------------------------------------------
    # CSV 저장
    # --------------------------------------------------------

    save_columns = [

        "rank",

        "appid",

        "game_name",

        "rating",

        "lightgbm_probability",

        "genre_preference_score"

    ]


    recommendations[
        save_columns
    ].to_csv(

        OUTPUT_CSV,

        index=False,

        encoding="utf-8-sig"

    )


    print()
    print("=" * 70)

    print(
        "추천 결과 저장 완료"
    )

    print(
        OUTPUT_CSV
    )

    print("=" * 70)


# ============================================================
# 실행
# ============================================================

if __name__ == "__main__":

    main()

Steam LightGBM 개인 맞춤 추천 시스템


Steam ID를 입력하세요:  76561198056237344


CSV 파일 로딩
사용자 데이터: 4,500개
TOP500 데이터: 95개

사용자 플레이 데이터: 30개

Steam 전체 게임 라이브러리 확인
Steam 전체 라이브러리: 306개

=== 사용자 장르 선호도 ===
Action         : 0.3598
Simulation     : 0.1568
RPG            : 0.1375
Adventure      : 0.1316
Strategy       : 0.1065
Casual         : 0.0671
Horror         : 0.0204
Survival       : 0.0204
Shooter        : 0.0000
Sports         : 0.0000
Racing         : 0.0000
Platformer     : 0.0000
Puzzle         : 0.0000

LightGBM 학습
학습 데이터: 95개
Positive: 13개
Negative: 82개

=== LightGBM 평가 ===
Accuracy : 0.6316
Precision: 0.0000
Recall   : 0.0000
F1-score : 0.0000
ROC-AUC  : 0.3125
PR-AUC   : 0.1505

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.80      0.75      0.77        16
           1       0.00      0.00      0.00         3

    accuracy                           0.63        19
   macro avg       0.40      0.38      0.39        19
weighted avg       0.67      0.63      0.65        19


=== Feature Importance =